In [0]:
dbutils.widgets.text("event_blob_path", "")

event_blob_path = dbutils.widgets.get("event_blob_path")

if not event_blob_path:
    raise ValueError(
        "event_blob_path is empty. Run through ADF."
    )

print("Event blob path:", event_blob_path)

In [0]:
relative_path = event_blob_path.replace("bronze/", "", 1)

bronze_path = (
    "abfss://bronze@stcommercesrc.dfs.core.windows.net/"
    + relative_path
)

print("Bronze path:")
print(bronze_path)

In [0]:
print("STEP 1 - Reading Bronze order file")

df_bronze = spark.read.parquet(bronze_path)

rows_read = df_bronze.count()

print("Rows read from Bronze:", rows_read)

df_bronze.printSchema()

display(df_bronze.limit(10))

In [0]:
from pyspark.sql.functions import col, trim, upper, initcap

print("STEP 20.1 - Cleaning Customer Addresses")

df_clean = (
    df_bronze
    .withColumn("address_type", upper(trim(col("address_type"))))
    .withColumn("address_line1", trim(col("address_line1")))
    .withColumn("address_line2", trim(col("address_line2")))
    .withColumn("city", initcap(trim(col("city"))))
    .withColumn("state", initcap(trim(col("state"))))
    .withColumn("postal_code", trim(col("postal_code")))
    .withColumn("country", upper(trim(col("country"))))
)

print("Rows before cleaning:", df_bronze.count())
print("Rows after cleaning:", df_clean.count())

display(df_clean.limit(10))

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

print("STEP 20.2 - Deduplicating Customer Addresses")

window_spec = (
    Window
    .partitionBy("address_id")
    .orderBy(
        col("updated_at").desc_nulls_last(),
        col("created_at").desc_nulls_last()
    )
)

df_ranked = df_clean.withColumn(
    "row_num",
    row_number().over(window_spec)
)

df_deduplicated = (
    df_ranked
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Rows before deduplication:", df_clean.count())
print("Rows after deduplication:", df_deduplicated.count())
print(
    "Duplicates removed:",
    df_clean.count() - df_deduplicated.count()
)

In [0]:
print("STEP 20.3 - Validating Customer Addresses")

df_valid = df_deduplicated.filter(
    col("address_id").isNotNull()
    & col("customer_id").isNotNull()
    & col("address_type").isNotNull()
    & (col("address_type") != "")
    & col("address_line1").isNotNull()
    & (col("address_line1") != "")
    & col("city").isNotNull()
    & (col("city") != "")
    & col("state").isNotNull()
    & (col("state") != "")
    & col("postal_code").isNotNull()
    & (col("postal_code") != "")
    & col("country").isNotNull()
    & (col("country") != "")
)

df_rejected = df_deduplicated.join(
    df_valid.select("address_id"),
    on="address_id",
    how="left_anti"
)

print("Deduplicated rows:", df_deduplicated.count())
print("Valid addresses:", df_valid.count())
print("Rejected addresses:", df_rejected.count())

display(df_rejected.limit(10))

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

print("STEP 21.2 - Fixing Customer Addresses MERGE")

# Map the Bronze/Silver input column to the target column
df_merge = df_valid.withColumn(
    "is_effective_default",
    col("is_default")
)

target_table = "ecommerce.silver.customer_addresses_external"
target = DeltaTable.forName(spark, target_table)

(
    target.alias("t")
    .merge(
        df_merge.alias("s"),
        "t.address_id = s.address_id"
    )
    .whenMatchedUpdate(set={
        "customer_id": "s.customer_id",
        "address_type": "s.address_type",
        "address_line1": "s.address_line1",
        "address_line2": "s.address_line2",
        "city": "s.city",
        "state": "s.state",
        "postal_code": "s.postal_code",
        "country": "s.country",
        "is_effective_default": "s.is_effective_default",
        "created_at": "s.created_at",
        "updated_at": "s.updated_at"
    })
    .whenNotMatchedInsert(values={
        "address_id": "s.address_id",
        "customer_id": "s.customer_id",
        "address_type": "s.address_type",
        "address_line1": "s.address_line1",
        "address_line2": "s.address_line2",
        "city": "s.city",
        "state": "s.state",
        "postal_code": "s.postal_code",
        "country": "s.country",
        "is_effective_default": "s.is_effective_default",
        "created_at": "s.created_at",
        "updated_at": "s.updated_at"
    })
    .execute()
)

print("Customer Addresses MERGE completed successfully.")

In [0]:
%sql
SELECT * FROM ecommerce.silver.customer_addresses_external

address_id,customer_id,address_type,address_line1,address_line2,city,state,postal_code,country,is_default,is_effective_default,created_at,updated_at
1,20953,HOME,"H.No. 433, Chaudry Street",,Gudivada,Maharashtra,001338,INDIA,true,true,2026-02-14T09:46:12Z,2026-06-08T09:46:12Z
3,13826,HOME,H.No. 559 Bassi Road,,Jorhat,Andhra Pradesh,618495,INDIA,true,true,2025-11-29T18:29:47Z,2026-03-28T18:29:47Z
5,22982,OTHER,"28/32, Raghavan Chowk",Near Bus Stand,Proddatur,Tamil Nadu,503056,INDIA,false,false,2023-04-12T20:12:25Z,2024-02-07T20:12:25Z
6,9116,HOME,"953, Randhawa Chowk",Apartment 101,Asansol,Gujarat,238849,INDIA,false,false,2024-08-24T02:20:28Z,2025-01-13T02:20:28Z
9,23912,OFFICE,"48/932, Edwin Path",Near Main Road,Khora,Gujarat,701543,INDIA,true,true,2021-06-29T23:12:49Z,2022-04-07T23:12:49Z
12,3310,OFFICE,"H.No. 103, Balay Zila",Block A,Madurai,Kerala,473829,INDIA,false,false,2025-11-26T05:27:16Z,2026-02-17T05:27:16Z
13,12131,OFFICE,763 Chatterjee Zila,Flat 202,Raichur,Maharashtra,670106,INDIA,true,true,2023-11-21T04:14:08Z,2024-09-27T04:14:08Z
15,8846,OTHER,"H.No. 326, Srivastava Chowk",Flat 202,Rewa,Delhi,260647,INDIA,true,true,2023-06-06T05:43:27Z,2023-10-01T05:43:27Z
16,1052,OFFICE,87/234 Bhardwaj Ganj,Near Bus Stand,Satara,Maharashtra,097882,INDIA,true,true,2021-06-26T17:14:54Z,2021-10-12T17:14:54Z
17,18586,OTHER,"191, Raval Ganj",2nd Floor,Jehanabad,Tamil Nadu,990916,INDIA,false,false,2026-08-05T09:17:45Z,2027-02-23T09:17:45Z
